# Load cases for each configuration

In [40]:
import sys
import numpy as np
sys.path.append('../')

from Material import Material
from Equilibrium import PointLoad, RunningLoad, EquilibriumEquation, Moment
from Analysis import WingBox
from cg_est import Wing, Fuselage, LandingGear, Propulsion, Weight
from constants import *

**Eliptical Lift Distribution**

$(z/b)^2 + (y/a)^2 = 1$

$2L / (\pi \cdot b) = a$

$y = a \cdot \sqrt{1 - (z/b)^2}$

**Equilibrium**

$\sum \vec{F} = \vec{0}$

$\sum \vec{M} = \vec{0}$

**Shear**

$q_{2} - q_1 = \int_{s1}^{s2}\frac{\delta q}{\delta s} ds = -\frac{V_y I_{yy} + V_x I_{xy}}{I_{xx}I_{yy} + I_{xy}^2}\int_{s1}^{s2}tyds - \frac{V_x I_{xx} + V_y I_{xy}}{I_{xx}I_{yy} + I_{xy}^2}\int_{s1}^{s2}txds$

**Bending**

$\sigma_z = \frac{M_x y}{I_{xx}} + \frac{M_y x}{I_{yy}}$

## Tandem Wing

In [68]:
config = 1

WoS = WS
Pmax = 8.77182
mProp = 400 / N_cruise
thickness = 3e-3
w_fus= 1.3; h_fus=1.6; l_fus=4
nmax = 3.02
b = (AR * S_front) ** 0.5

w = Weight(95, Wing(mTO, S_front, S_back, 1.5*nmax, AR, [0.4, 3.6], config),
           Fuselage(mTO, Pmax, l_fus, 5, l_fus/2, config),
          LandingGear(mTO, l_fus/2),
          Propulsion(N_cruise, [mProp]*N_cruise, pos_prop=[3.6]*int(N_cruise/2) + [0.4]*int(N_cruise/2)),
          cargo_m, cargo_p, Bat_mass, l_fus/2, [0.8, 1.3, 1.3, 2.5, 2.5])

ukloads = [PointLoad([1, 0, 0], [0, 0, 0]), PointLoad([0, 1, 0], [0, 0, 0]), PointLoad([0, 0, 1], [0, 0, 0]),
          Moment([1, 0, 0]), Moment([0, 1, 0]), Moment([0, 0, 1])]

Thrusts = [PointLoad([0, Max_T_engine, 0], [0, 0, d]) for d in np.linspace(0, b/2, round(N_hover / 4))]
distrWeight = RunningLoad([[0]*3, [ -w.wing.get_weight()[0] * 9.81 / b ]*3], [0, b/4, b/2], 2)

box = WingBox(thickness, 0.8 * 0.17 * c_r, 0.8 * c_r)

eql = EquilibriumEquation(kloads=[distrWeight] + Thrusts, ukloads=ukloads)
eql.SetupEquation()
Fx, Fy, Fz, Mx, My, Mz = eql.SolveEquation()

print("VTOL: τ, σ, Y [MPa]")
print(tauVTOL := box.tau(box.b/2, 0, Fx, Fy, Mz)*1e-6)
print(oVTOL := box.o(0, box.h/2, -Mx, My)*1e-6)
print(YVTOL := (oVTOL ** 2 + 3 * tauVTOL ** 2) ** 0.5)

rho = 1.205
toc = 0.17
box = WingBox(thickness, 0.8 * c_r, 0.8 * 0.17 * c_r)

liftWing = 2 * (9.81 * mTO * nmax * 1.5 / 2) / ( np.pi * b )
zpos = np.linspace(0, b/2, 1000)
Lift = RunningLoad([[0] * len(zpos), [liftWing * ( 1 - (z/b) ** 2 ) ** 0.5 for z in zpos]], zpos, axis=2)
Drag = RunningLoad([[liftWing / LD_ratio * ( 1 - (z/b) ** 2 ) ** 0.5 for z in zpos], [0] * len(zpos)], zpos, axis=2, poa=(-0.25 * MAC1, 0))
ADMoment = Moment([0, 0, Cm_ac_front * 0.5 * rho * V_cruise ** 2 * S_front * MAC1])
Thrusts = [PointLoad([-liftWing * 4 / (LD_ratio*N_cruise), 0, 0],
                     [0.45 * MAC1, toc * MAC1 * 0.5, d]) for d in np.linspace(0, b/2, round(N_cruise / 4))]

cruise = EquilibriumEquation(kloads=[Lift, Drag, ADMoment, distrWeight] + Thrusts,
                            ukloads=ukloads)
cruise.SetupEquation()
Rfx, Rfy, Rfz, Rmx, Rmy, Rmz = cruise.SolveEquation()
print("\nCruise: τ, σ, Y [MPa]")
print(taucr := box.tau(-box.b/2, 0, Rfx, Rfy, Rmz)*1e-6)
print(ocr := box.o(-box.b/2, -box.h/2, -Rmx, Rmy)*1e-6)
print(Ycr := (ocr ** 2 + 3 * taucr ** 2) ** 0.5)
aluminum = Material.load(material='Al 6061', Condition='T6')
print("\nFatigue:")
print(fatigueLife := aluminum.ParisFatigueN(ocr*1e6 - oVTOL*1e6, box.b, 0.375 * 1.2e-3, box.t/2)*1e-6)
print(bucklingStress := aluminum.buckling(box.b, box.t)*1e-6)

VTOL: τ, σ, Y [MPa]
-3.502632737146191
-22.6410480372466
23.43976033364874

Cruise: τ, σ, Y [MPa]
30.19891156258223
69.18706995548472
86.73392316573212

Fatigue:
9.889447074741815
6.029288768866562


## Box Wing

In [72]:
config = 2

WoS = WS
Pmax = 8.77182
mProp = 400 / N_cruise
thickness = 3e-3
w_fus= 1.3; h_fus=1.6; l_fus=4
nmax = 3.02
b = (AR * S_front) ** 0.5

w = Weight(95, Wing(mTO, S_front, S_back, 1.5*nmax, AR, [0.4, 3.6], config),
           Fuselage(mTO, Pmax, l_fus, 5, l_fus/2, config),
          LandingGear(mTO, l_fus/2),
          Propulsion(N_cruise, [mProp]*N_cruise, pos_prop=[3.6]*int(N_cruise/2) + [0.4]*int(N_cruise/2)),
          cargo_m, cargo_p, Bat_mass, l_fus/2, [0.8, 1.3, 1.3, 2.5, 2.5])

ukloads = [PointLoad([1, 0, 0], [0, 0, 0]), PointLoad([0, 1, 0], [0, 0, 0]), PointLoad([0, 0, 1], [0, 0, 0]),
          Moment([1, 0, 0]), Moment([0, 1, 0]), Moment([0, 0, 1])]

Thrusts = [PointLoad([0, Max_T_engine, 0], [0.45 * MAC1, 0, d]) for d in np.linspace(0.1 * b/2, 0.6 * b/2, round(N_hover / 4))]
distrWeight = RunningLoad([[0]*3, [ -w.wing.get_weight()[0] * 9.81 / b ]*3], [0, b/4, b/2], 2)

eql = EquilibriumEquation(kloads=[distrWeight] + Thrusts, ukloads=ukloads)
eql.SetupEquation()
Fx, Fy, Fz, Mx, My, Mz = eql.SolveEquation()
box = WingBox(thickness, 0.8 * c_r, 0.8 * 0.17 * c_r)
print("VTOL: τ, σ, Y [MPa]")
print(tauVTOL := box.tau(box.b/2, 0, Fx, Fy, Mz)*1e-6)
print(oVTOL := box.o(0, box.h/2, -Mx, My)*1e-6)
print(YVTOL := (oVTOL ** 2 + 3 * tauVTOL ** 2) ** 0.5)

rho = 1.205
toc = 0.17

liftWing = 2 * (9.81 * mTO * nmax * 1.5 / 2) / ( np.pi * b )
zpos = np.linspace(0, b/2, 1000)
Lift = RunningLoad([[0] * len(zpos), [liftWing * ( 1 - (z/b) ** 2 ) ** 0.5 for z in zpos]], zpos, axis=2)
Drag = RunningLoad([[liftWing / LD_ratio * ( 1 - (z/b) ** 2 ) ** 0.5 for z in zpos], [0] * len(zpos)], zpos, axis=2, poa=(-0.25 * MAC1, 0))
ADMoment = Moment([0, 0, Cm_ac_front * 0.5 * rho * V_cruise ** 2 * S_front * MAC1])
Thrusts = [PointLoad([-liftWing * 4 / (LD_ratio*N_cruise), 0, 0],
                     [0.45 * MAC1, toc * MAC1 * 0.5, d]) for d in np.linspace(0.1 * b/2, 0.6 * b/2, round(N_cruise / 4))]

cruise = EquilibriumEquation(kloads=[Lift, Drag, ADMoment, distrWeight] + Thrusts,
                            ukloads=ukloads)
cruise.SetupEquation()
Rfx, Rfy, Rfz, Rmx, Rmy, Rmz = cruise.SolveEquation()
print("\nCruise: τ, σ, Y [MPa]")
print(taucr := box.tau(-box.b/2, 0, Rfx, Rfy, Rmz)*1e-6)
print(ocr := box.o(-box.b/2, -box.h/2, -Rmx, Rmy)*1e-6)
print(Ycr := (ocr ** 2 + 3 * taucr ** 2) ** 0.5)
aluminum = Material.load(material='Al 6061', Condition='T6')
print("\nFatigue & Buckling:")
print(fatigueLife := aluminum.ParisFatigueN(taucr*1e6 - tauVTOL*1e6, box.b, 0.375 * 1.2e-3, box.t/2)*1e-6)
print(bucklingStress := aluminum.buckling(box.b, box.t)*1e-6)

VTOL: τ, σ, Y [MPa]
-134.68897319967516
-43.32933442966012
237.27787449921615

Cruise: τ, σ, Y [MPa]
30.19891156258223
69.32382729038798
86.84305273815245

Fatigue & Buckling:
1.7081768077541686
6.029288768866562


## Single Wing

In [69]:
config = 3

WoS = WS
Pmax = 8.77182
mProp = 400 / N_cruise
thickness = 3e-3
w_fus= 1.3; h_fus=1.6; l_fus=4
nmax = 3.02
b = (AR * S_front) ** 0.5

w = Weight(95, Wing(mTO, S_front, S_back, 1.5*nmax, AR, [0.4, 3.6], config),
           Fuselage(mTO, Pmax, l_fus, 5, l_fus/2, config),
          LandingGear(mTO, l_fus/2),
          Propulsion(N_cruise, [mProp]*N_cruise, pos_prop=[3.6]*int(N_cruise/2) + [0.4]*int(N_cruise/2)),
          cargo_m, cargo_p, Bat_mass, l_fus/2, [0.8, 1.3, 1.3, 2.5, 2.5])

ukloads = [PointLoad([1, 0, 0], [0, 0, 0]), PointLoad([0, 1, 0], [0, 0, 0]), PointLoad([0, 0, 1], [0, 0, 0]),
          Moment([1, 0, 0]), Moment([0, 1, 0]), Moment([0, 0, 1])]

Thrust = PointLoad([0, Max_T_wing_engine, 0], [0, 0, 0.1 * b])
distrWeight = RunningLoad([[0]*3, [ -w.wing.get_weight()[0] * 9.81 / b ]*3], [0, b/4, b/2], 2)

box = WingBox(thickness, 0.8 * c_r, 0.8 * 0.17 * c_r)

eql = EquilibriumEquation(kloads=[distrWeight] + Thrusts, ukloads=ukloads)
eql.SetupEquation()
Fx, Fy, Fz, Mx, My, Mz = eql.SolveEquation()

print("VTOL: τ, σ, Y [MPa]")
print(tauVTOL := box.tau(box.b/2, 0, Fx, Fy, Mz)*1e-6)
print(oVTOL := box.o(0, box.h/2, -Mx, My)*1e-6)
print(YVTOL := (oVTOL ** 2 + 3 * tauVTOL ** 2) ** 0.5)

rho = 1.205
toc = 0.17
box = WingBox(thickness, 0.8 * c_r, 0.8 * 0.17 * c_r)

liftWing = 2 * (9.81 * mTO * nmax * 1.5) / ( np.pi * b )
zpos = np.linspace(0, b/2, 1000)
Lift = RunningLoad([[0] * len(zpos), [liftWing * ( 1 - (z/b) ** 2 ) ** 0.5 for z in zpos]], zpos, axis=2)
Drag = RunningLoad([[liftWing / LD_ratio * ( 1 - (z/b) ** 2 ) ** 0.5 for z in zpos], [0] * len(zpos)], zpos, axis=2, poa=(-0.25 * MAC1, 0))
ADMoment = Moment([0, 0, Cm_ac_front * 0.5 * rho * V_cruise ** 2 * S_front * MAC1])
wEngine = PointLoad([0, -981, 0], [0, 0, 0.1 * b])
cruise = EquilibriumEquation(kloads=[Lift, Drag, ADMoment, distrWeight, wEngine],
                            ukloads=ukloads)
cruise.SetupEquation()
Rfx, Rfy, Rfz, Rmx, Rmy, Rmz = cruise.SolveEquation()
print("\nCruise: τ, σ, Y [MPa]")
print(taucr := box.tau(-box.b/2, 0, Rfx, Rfy, Rmz)*1e-6)
print(ocr := box.o(-box.b/2, -box.h/2, -Rmx, Rmy)*1e-6)
print(Ycr := (ocr ** 2 + 3 * taucr ** 2) ** 0.5)
aluminum = Material.load(material='Al 6061', Condition='T6')
print("\nFatigue:")
print(fatigueLife := aluminum.ParisFatigueN(ocr*1e6 + oVTOL*1e6, box.b, 0.375 * 1.2e-3, box.t/2)*1e-6)
print(bucklingStress := aluminum.buckling(box.b, box.t)*1e-6)

VTOL: τ, σ, Y [MPa]
0.46613196947101726
3.5679902661623166
3.658195125785095

Cruise: τ, σ, Y [MPa]
47.677809736396924
140.6436860817861
163.0955764527419

Fatigue:
2.5532761335215026
6.029288768866562
